# Task 2 – Violation features
**Input:** `food_clean.csv`   
**Output:** `joud_features.csv`, the violation columns, joined with Maram's columns at the end of the week.

In [1]:
import pandas as pd

We only need pandas here.

In [2]:
df = pd.read_csv('food_clean.csv')
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'])
print(df.shape)

(212463, 22)


We read the clean file, the date becomes text again when saved to CSV, so we change it back to a real date.

In [3]:
# turning the violation text into a list of numbers, "38. INSECTS..." -> 38
def get_numbers(text):
    if pd.isnull(text):
        return []
    numbers = []
    for v in text.split(' | '):
        num = v.strip().strip('"').split('.')[0]
        if num.isdigit():
            numbers.append(int(num))
    return numbers

df['Viol_Numbers'] = df['Violations'].apply(get_numbers)
df[['Violations', 'Viol_Numbers']].head()

,Violations,Viol_Numbers
0,NaN,[]
1,57. ALL FOOD EMPLOYEES HAVE FOOD HANDLER TRAIN...,[57]
2,NaN,[]
3,"1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...","[1, 2, 47, 51, 52, 54, 55, 55, 56, 58]"
4,NaN,[]


**1.** Each violation in the text starts with its number and a dot, and violations are separated by ` | `. We keep only the numbers, and an empty text (no violations) gives an empty list.

In [4]:
# number of violations in each inspection
df['Num_Violations'] = df['Viol_Numbers'].apply(len)
df['Num_Violations'].describe()

count    212463.000000
mean          3.788848
std           3.284113
min           0.000000
25%           1.000000
50%           3.000000
75%           5.000000
max          34.000000
Name: Num_Violations, dtype: float64

**2.** The number of violations is the length of the list. For all years 2010–2022: mean = 3.79, median = 3, max = 34.

In [5]:
# one function for all topic columns
change_date = pd.Timestamp('2018-07-01')

def has_topic(row, old, new):
    if row['Inspection Date'] < change_date:
        codes = old
    else:
        codes = new
    for n in row['Viol_Numbers']:
        if n in codes:
            return 1
    return 0

**3.** The same topic has different numbers before and after 1 July 2018. This function checks the date of the inspection, then looks for the old numbers (before) or the new numbers (after), it returns 1 if the topic is found, 0 if not.
`apply(axis=1)` is new thing for us: it sends a whole row to the function, so the function can use two columns (date and numbers)

In [6]:
# old numbers = before 1 July 2018, new numbers = after (from violations_titles_before/after.csv)
df['has_rodents'] = df.apply(has_topic, axis=1, old=[13, 18], new=[38])
df['has_temperature'] = df.apply(has_topic, axis=1, old=[2, 3], new=[18, 19, 20, 21, 22, 33])
df['has_handwashing'] = df.apply(has_topic, axis=1, old=[6, 12], new=[8, 9, 10])
df['has_toxic'] = df.apply(has_topic, axis=1, old=[25], new=[28])
df['has_water_sewage'] = df.apply(has_topic, axis=1, old=[9, 10], new=[50, 52])
df['has_dirty_surfaces'] = df.apply(has_topic, axis=1, old=[33, 34, 35], new=[49, 55])
df['has_thermometers'] = df.apply(has_topic, axis=1, old=[40], new=[36])
df['has_food_handler'] = df.apply(has_topic, axis=1, old=[45], new=[57])
df['has_smoking'] = df.apply(has_topic, axis=1, old=[70], new=[62])
df['has_summary_report'] = df.apply(has_topic, axis=1, old=[28], new=[61])

**Crosswalk (old → new), matched by the titles in the violation-list files:**

| Topic | Old numbers | New numbers |
|---|---|---|
| Rodents / insects | 13 (infestation), 18 (rodent proofing) | 38 |
| Temperature | 2, 3 | 18, 19, 20, 21, 22, 33 |
| Hand washing | 6 (hands, bare hand), 12 (sinks) | 8, 9, 10 |
| Toxic items | 25 | 28 |
| Water / sewage | 9 (water), 10 (sewage) | 50, 52 |
| Dirty surfaces | 33, 34, 35 | 49, 55 |
| Thermometers | 40 | 36 |
| Food handler training | 45 | 57 |
| Smoking (new #62 = Clean Indoor Air Ordinance, Chicago's no-smoking law) | 70 | 62 |
| Inspection report / summary displayed | 28 | 61 |

The two lists are not one-to-one, so this is an approximation.

We removed new #51 (plumbing / backflow) from water / sewage: in the old list plumbing was part of #38 (ventilation, plumbing), so #51 has no match in old #9 or #10.

In [7]:
# share of inspections with each topic, per period
topics = ['has_rodents', 'has_temperature', 'has_handwashing', 'has_toxic', 'has_water_sewage', 'has_dirty_surfaces',
          'has_thermometers', 'has_food_handler', 'has_smoking', 'has_summary_report']
study = df[df['Year'] >= 2015]   # only the years we study
study.groupby('Period')[topics].mean().round(3)

,has_rodents,has_temperature,has_handwashing,has_toxic,has_water_sewage,has_dirty_surfaces,has_thermometers,has_food_handler,has_smoking,has_summary_report
Period,,,,,,,,,,
Grace (90 days),0.153,0.109,0.198,0.004,0.035,0.558,0.115,0.163,0.002,0.015
Late transition,0.155,0.077,0.237,0.004,0.033,0.553,0.122,0.181,0.002,0.012
Post,0.161,0.071,0.206,0.004,0.033,0.532,0.084,0.132,0.002,0.020
Pre,0.109,0.081,0.037,0.001,0.013,0.676,0.086,0.116,0.002,0.011


**4.** If a topic's share jumps a lot between Pre and Post, the matching may be wrong (or the rule really changed). We use only the study years (2015–2022), so Pre means 2015 – June 2018, like in the tests.

In [8]:
# which new number makes the jump? (share of Post inspections with each number)
after = study[study['Period'] == 'Post']
for n in [8, 9, 10, 50, 51, 52]:
    print(n, after['Viol_Numbers'].apply(lambda nums: n in nums).mean().round(3))

8 0.006
9 0.005
10 0.201
50 0.013
51 0.232
52 0.021


**Check of the jumps.** Hand washing goes from about 4% (Pre) to about 21% (Post), almost all of it comes from new #10 “Adequate handwashing sinks properly supplied and accessible” (20.1% of Post inspections), while #8 and #9 are under 1%, the new code checks sink supplies (soap, towels) much more strictly, so this is a real rule change, not a matching mistake, we keep the column, but its meaning is slightly different before and after 2018.

New #51 “Plumbing installed; proper backflow devices” appears in 23.2% of Post inspections. this is why water/sewage jumped before; after removing #51 it is much closer to the Pre level (1.3% vs 3.3%).

Temperature and toxic items stay about the same before and after, so their matching works well. Rodents rise a little (11% → 16%) and dirty surfaces fall (68% → 53%). Hand washing rises a lot because of the new #10. This supports H1: the mix of violations changes after the 2018 code.

## Severity levels (outside information)
The CSV only gives the violation number and title. It never says **how serious** a violation is. That information comes from outside the dataset, from the official City of Chicago documents:
- **Before 1 July 2018** (old code): violations 1–14 are **Critical**, 15–29 are **Serious**, 30–44 and 70 are **Minor** (Chicago Data Portal dataset description; CDPH *Chicago Food Code Major Changes*).
- **After 1 July 2018** (new code): violations are **Priority (P)**, **Priority Foundation (PF)** or **Core**, and the inspector writes the level in the comment, e.g. "PRIORITY VIOLATION 7-38-005" (CDPH *Chicago Food Code Major Changes*; CDPH Inspection FAQ 2019).

We use this outside information to count, for each inspection, how many **critical** violations it has (old: Critical, 1–14; new: Priority) and how many **serious** ones (old: Serious, 15–29; new: Priority Foundation). The City of Chicago's own model kept these two levels separate (`pastCritical`, `pastSerious`), so we do the same.

In [9]:
# check: do the comments after 1 July 2018 name the level?
after_text = df[df['Inspection Date'] >= change_date]['Violations'].dropna()
print('contains PRIORITY FOUNDATION:', after_text.str.contains('PRIORITY FOUNDATION').mean().round(3))
print('contains PRIORITY VIOLATION:', after_text.str.contains('PRIORITY VIOLATION').mean().round(3))
print('contains PRIORITY (any):', after_text.str.contains('PRIORITY').mean().round(3))
print('contains CORE:', after_text.str.contains('CORE').mean().round(3))

contains PRIORITY FOUNDATION: 0.598
contains PRIORITY VIOLATION: 0.144
contains PRIORITY (any): 0.638
contains CORE: 0.075


Before we build the column we check that the new-code comments really contain the level words. These numbers are the share of post-2018 inspections (with violations) whose text contains each word. **Result:** after July 2018, 63.8% of the inspections with violations mention PRIORITY in the comments (59.8% PRIORITY FOUNDATION, 14.4% PRIORITY VIOLATION, 7.5% CORE). So the inspectors write the level in the comment, and we can use it to build the severity columns.

In [10]:
# split each inspection's violations into the old/new severity levels
def get_level(v, date):
    num = v.strip().strip('"').split('.')[0]
    if not num.isdigit():
        return 'other'
    num = int(num)
    if date < change_date:
        if num <= 14:                       # old code: 1-14 Critical
            return 'critical'
        elif num <= 29:                     # old code: 15-29 Serious
            return 'serious'
        else:
            return 'other'                  # Minor
    else:
        if num in [59, 60]:                 # "previous violation corrected", not a new problem
            return 'other'
        if ' - Comments:' not in v:
            return 'other'
        comment = v.split(' - Comments:')[1]
        if 'PRIORITY FOUNDATION' in comment:  # new code: Priority Foundation ~ Serious
            return 'serious'
        elif 'PRIORITY' in comment:           # new code: Priority ~ Critical
            return 'critical'
        else:
            return 'other'                  # Core

def count_level(row, level):
    text = row['Violations']
    if pd.isnull(text):
        return 0
    count = 0
    for v in text.split(' | '):
        if get_level(v, row['Inspection Date']) == level:
            count += 1
    return count

df['Num_Critical'] = df.apply(count_level, axis=1, level='critical')
df['Num_Serious'] = df.apply(count_level, axis=1, level='serious')
df[['Num_Critical', 'Num_Serious']].describe()

,Num_Critical,Num_Serious
count,212463.000000,212463.000000
mean,0.212757,0.575164
std,0.575329,1.108861
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,1.000000
max,8.000000,16.000000


**Num_Critical and Num_Serious.** `get_level` decides the level of one violation. Before the change it uses the number (1–14 Critical, 15–29 Serious, the rest Minor). After the change it reads the inspector's comment: "PRIORITY FOUNDATION" = serious, otherwise "PRIORITY" = critical, otherwise Core. We check "PRIORITY FOUNDATION" first because it also contains the word PRIORITY. New #59 and #60 are skipped, because they only say an earlier violation was corrected. `count_level` counts the violations of one level in an inspection. **Result (2010–2022):** Num_Critical has a mean of 0.21 and a max of 8; Num_Serious has a mean of 0.58 and a max of 16. Most inspections have no critical violation (the median is 0).

In [11]:
# check: do failed inspections have more critical and serious violations?
study = df[df['Year'] >= 2015]
study.groupby('Results')[['Num_Critical', 'Num_Serious']].mean().round(2)

,Num_Critical,Num_Serious
Results,,
Fail,0.57,1.87
Pass,0.00,0.01
Pass w/ Conditions,0.41,1.44


If the levels make sense, **Fail** inspections should have clearly more critical / serious violations than **Pass** inspections, because a restaurant fails when a serious problem cannot be fixed during the visit. **Result:** Pass inspections have almost no critical or serious violations (0.00 and 0.01), which matches the official definition of Pass. Fail inspections have the most (0.57 critical, 1.87 serious) and Pass w/ Conditions is in between (0.41 and 1.44). This confirms that the severity levels were built correctly.

In [12]:
# check: do the columns look similar before and after the change?
study = df[df['Year'] >= 2015]
study.groupby('Period')[['Num_Critical', 'Num_Serious']].mean().round(2)

,Num_Critical,Num_Serious
Period,,
Grace (90 days),0.30,2.29
Late transition,0.21,1.92
Post,0.20,0.91
Pre,0.24,0.36


Average number of critical and serious violations per period. A big jump at 2018 would mean the old and new levels are not measured in exactly the same way. **Result:** critical violations stay almost the same in all periods (Pre 0.24, Grace 0.30, Late transition 0.21, Post 0.20), so the old Critical level and the new Priority level are comparable. Serious violations jump from 0.36 (Pre) to 2.29 in the grace period, 1.92 in the late transition and 0.91 after. The new Priority Foundation level covers many more problems than the old Serious level. This supports H1, and it matters for the model: Num_Serious does not mean exactly the same thing before and after 2018.

## Previous visit (for the model, without leakage)
The model must not see the violations of the **same** visit, because they are written during the visit and the result is decided from them (leakage). It may use the violations of the **previous** visit of the same business, which are known before the new visit. The City of Chicago model did the same: `pastCritical` and `pastSerious` = any critical / serious violation at the last visit.

In [13]:
df = df.sort_values(by=['Business_ID', 'Inspection Date'])

We sort each business's inspections from oldest to newest, so that "the row before" is the previous visit of the same business.

In [14]:
df['Has_Critical'] = df['Num_Critical'].apply(lambda x: 1 if x > 0 else 0)
df['Has_Serious'] = df['Num_Serious'].apply(lambda x: 1 if x > 0 else 0)

Like Chicago, we turn the counts into yes/no: 1 if the visit had at least one critical (or serious) violation.

In [15]:
df['Prev_Critical'] = df.groupby('Business_ID')['Has_Critical'].shift(1).fillna(0)
df['Prev_Serious'] = df.groupby('Business_ID')['Has_Serious'].shift(1).fillna(0)
df['Prev_Num_Violations'] = df.groupby('Business_ID')['Num_Violations'].shift(1).fillna(0)
df[['Business_ID', 'Inspection Date', 'Has_Critical', 'Prev_Critical', 'Num_Violations', 'Prev_Num_Violations']].head(10)

,Business_ID,Inspection Date,Has_Critical,Prev_Critical,Num_Violations,Prev_Num_Violations
198578,1000049_PEREZ MEXICAN FOOD,2010-10-18,0,0.0,6,0.0
198241,1000049_PEREZ MEXICAN FOOD,2010-10-25,0,0.0,0,6.0
180267,1000049_PEREZ MEXICAN FOOD,2011-11-15,1,0.0,3,0.0
168293,1000049_PEREZ MEXICAN FOOD,2012-08-15,0,1.0,4,3.0
150563,1000049_PEREZ MEXICAN FOOD,2013-09-09,0,0.0,5,4.0
150092,1000049_PEREZ MEXICAN FOOD,2013-09-17,0,0.0,4,5.0
141090,1000049_PEREZ MEXICAN FOOD,2014-04-04,1,0.0,10,4.0
140472,1000049_PEREZ MEXICAN FOOD,2014-04-16,0,1.0,5,10.0
128375,1000049_PEREZ MEXICAN FOOD,2014-11-24,0,0.0,3,5.0
123528,1000049_PEREZ MEXICAN FOOD,2015-03-13,1,0.0,9,3.0


`shift(1)` inside each business moves the values down one row, so each visit gets the values of the previous visit (same method as `Prev_Fail` in Maram's notebook). The first visit of a business has no previous visit, so it gets 0; Maram's `First_Inspection` column marks these rows. (Check in the table: Prev_ of each row = the value of the row above, for the same Business_ID.)

In [16]:
# whole history: share of earlier visits with a critical / serious violation
num_past = df.groupby('Business_ID').cumcount()
df['Past_Critical_Rate'] = ((df.groupby('Business_ID')['Has_Critical'].cumsum() - df['Has_Critical']) / num_past).fillna(0)
df['Past_Serious_Rate'] = ((df.groupby('Business_ID')['Has_Serious'].cumsum() - df['Has_Serious']) / num_past).fillna(0)
df[['Business_ID', 'Inspection Date', 'Has_Critical', 'Past_Critical_Rate']].head(10)

,Business_ID,Inspection Date,Has_Critical,Past_Critical_Rate
198578,1000049_PEREZ MEXICAN FOOD,2010-10-18,0,0.000000
198241,1000049_PEREZ MEXICAN FOOD,2010-10-25,0,0.000000
180267,1000049_PEREZ MEXICAN FOOD,2011-11-15,1,0.000000
168293,1000049_PEREZ MEXICAN FOOD,2012-08-15,0,0.333333
150563,1000049_PEREZ MEXICAN FOOD,2013-09-09,0,0.250000
150092,1000049_PEREZ MEXICAN FOOD,2013-09-17,0,0.200000
141090,1000049_PEREZ MEXICAN FOOD,2014-04-04,1,0.166667
140472,1000049_PEREZ MEXICAN FOOD,2014-04-16,0,0.285714
128375,1000049_PEREZ MEXICAN FOOD,2014-11-24,0,0.250000
123528,1000049_PEREZ MEXICAN FOOD,2015-03-13,1,0.222222


Like `Past_Fail_Rate` in Maram's notebook: the share of the **earlier** visits of the business that had a critical (or serious) violation. `cumcount()` counts the earlier visits, `cumsum()` adds up the earlier critical visits (we subtract the current visit so only the past counts). A rate is fair to new and old businesses. The first visit gets 0.

In [17]:
# check: are businesses with a critical violation last time more likely to fail now?
study = df[df['Year'] >= 2015]
pd.crosstab(study['Prev_Critical'], study['Fail'], normalize='index').round(3)

Fail,0,1
Prev_Critical,,
0.0,0.769,0.231
1.0,0.817,0.183


Fail rate for businesses with and without a critical violation at the previous visit (2015–2022). **Result:** businesses with a critical violation last time fail **less** often now (**18.3%**) than businesses without one (**23.1%**). This is probably caused by re-inspections: after a critical violation the city comes back soon, and the business has usually fixed the problem. We check this in the next cell.

In [18]:
# the same check without re-inspections (same rule as Is_Reinspection in Maram's notebook)
normal = study[~study['Inspection Type'].str.contains(r're-?insp', case=False, na=False)]
pd.crosstab(normal['Prev_Critical'], normal['Fail'], normalize='index').round(3)

Fail,0,1
Prev_Critical,,
0.0,0.740,0.260
1.0,0.713,0.287


The same check for normal visits only (no re-inspections). **Result:** now the pattern turns around: businesses with a critical violation last time fail **more** often (**28.7%**) than businesses without one (**26.0%**). So the surprising result above was caused by re-inspections, and Prev_Critical is a useful feature for normal visits (the same as Prev_Fail in Maram's notebook).

In [19]:
cols = ['Inspection ID', 'Num_Violations', 'Num_Critical', 'Num_Serious',
        'Prev_Critical', 'Prev_Serious', 'Prev_Num_Violations',
        'Past_Critical_Rate', 'Past_Serious_Rate'] + topics
df[cols].to_csv('joud_features.csv', index=False)
print(df[cols].shape)

(212463, 19)


We save only the ID and the new columns. In Task 3 this file is joined with `food_clean.csv` and Maram's file on `Inspection ID`. The columns of the **same** visit (Num_Violations, Num_Critical, Num_Serious, has_...) are for the statistical tests (H1); only the **Prev_** columns go into the model.